In [2]:
# =========================
# BLEU CORREGIDO (sentence-level con smoothing + corpus-level) — AUTOCONTENIDO
# =========================
import math, collections, re

TOKEN_RE = re.compile(r"\w+(?:'\w+)*|[^\w\s]")
def tokenize(text):
    return TOKEN_RE.findall(text)

def ngram_counts(tokens, n):
    return collections.Counter(tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1))

def sentence_bleu_smoothed(reference, hypothesis, max_n=4, tok_fn=None):
    tok_fn = tok_fn or tokenize
    ref_tokens = tok_fn(reference)
    hyp_tokens = tok_fn(hypothesis)
    if len(hyp_tokens) == 0:
        return 0.0
    n_max = min(max_n, len(hyp_tokens))
    precisions = []
    for n in range(1, n_max + 1):
        ref_ngrams = ngram_counts(ref_tokens, n)
        hyp_ngrams = ngram_counts(hyp_tokens, n)
        overlap = sum(min(c, ref_ngrams.get(g, 0)) for g, c in hyp_ngrams.items())
        total = sum(hyp_ngrams.values())
        if overlap == 0 and total > 0:
            overlap, total = overlap + 1, total + 1
        precisions.append(overlap / total if total > 0 else 0.0)
    if min(precisions) <= 0:
        return 0.0
    geo_mean = math.exp(sum(math.log(p) for p in precisions) / len(precisions))
    bp = 1.0 if len(hyp_tokens) > len(ref_tokens) else math.exp(1 - len(ref_tokens) / max(len(hyp_tokens), 1))
    return geo_mean * bp

def corpus_bleu(references, hypotheses, max_n=4, tok_fn=None):
    tok_fn = tok_fn or tokenize
    overlap_n = [0] * max_n
    total_n = [0] * max_n
    ref_len, hyp_len = 0, 0
    for ref, hyp in zip(references, hypotheses):
        r, h = tok_fn(ref), tok_fn(hyp)
        ref_len += len(r); hyp_len += len(h)
        for n in range(1, max_n + 1):
            rn, hn = ngram_counts(r, n), ngram_counts(h, n)
            overlap_n[n-1] += sum(min(c, rn.get(g, 0)) for g, c in hn.items())
            total_n[n-1] += sum(hn.values())
    precisions = [(overlap_n[i] + 1) / (total_n[i] + 1) if total_n[i] > 0 else 0.0 for i in range(max_n)]
    geo_mean = math.exp(sum(math.log(p) for p in precisions) / max_n) if min(precisions) > 0 else 0.0
    bp = 1.0 if hyp_len > ref_len else math.exp(1 - ref_len / max(hyp_len, 1))
    return geo_mean * bp, precisions, bp

# --- Autotest ---
print("=== Prueba del fix ===")
casos = [
    ("Pregúntale.", "Pregúntale ."),
    ("No.", "No ."),
    ("Ayúdame.", "Ayúdame ."),
    ("Gracias.", "De nada."),
]
for ref, hyp in casos:
    print(f"  ref={ref!r:20s} hyp={hyp!r:20s} -> BLEU={sentence_bleu_smoothed(ref, hyp):.4f}")

=== Prueba del fix ===
  ref='Pregúntale.'        hyp='Pregúntale .'       -> BLEU=1.0000
  ref='No.'                hyp='No .'               -> BLEU=1.0000
  ref='Ayúdame.'           hyp='Ayúdame .'          -> BLEU=1.0000
  ref='Gracias.'           hyp='De nada.'           -> BLEU=0.3816


In [3]:
# =========================
# CARGA DEL MODELO (época 14, OpenSubtitles) + ARQUITECTURA
# =========================
import os, json, math, time
import torch, torch.nn as nn, torch.nn.functional as F
import pandas as pd
from google.colab import drive

drive.mount("/content/drive")
PROJ = "/content/drive/MyDrive/Proyecto_traductor"
VOCAB_DIR_1M = os.path.join(PROJ, "vocabularios_opensubtitles_1m")
MODEL_DIR_1M = os.path.join(PROJ, "modelos_opensubtitles_1m")
SPLIT_DIR_1M = os.path.join(PROJ, "data", "opensubtitles_1m", "splits")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo:", DEVICE)

with open(os.path.join(VOCAB_DIR_1M, "vocab_english.json"), encoding="utf-8") as f: stoi_en = json.load(f)
with open(os.path.join(VOCAB_DIR_1M, "vocab_spanish.json"), encoding="utf-8") as f: stoi_es = json.load(f)
with open(os.path.join(MODEL_DIR_1M, "model_config.json"), encoding="utf-8") as f: model_cfg = json.load(f)
itos_es = {v: k for k, v in stoi_es.items()}
PAD_IDX, UNK_IDX, SOS_IDX, EOS_IDX = 0, 1, 2, 3

# --- Arquitectura (idéntica a la de entrenamiento) ---
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term); pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x): return x + self.pe[:, :x.size(1)]

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        self.d_k = d_model // n_heads; self.n_heads = n_heads
        self.w_q = nn.Linear(d_model, d_model); self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model); self.w_o = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)
    def forward(self, q, k, v, mask=None):
        B = q.size(0)
        Q = self.w_q(q).view(B, -1, self.n_heads, self.d_k).transpose(1, 2)
        K = self.w_k(k).view(B, -1, self.n_heads, self.d_k).transpose(1, 2)
        V = self.w_v(v).view(B, -1, self.n_heads, self.d_k).transpose(1, 2)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None: scores = scores.masked_fill(mask == 0, float('-inf'))
        attn = self.dropout(torch.softmax(scores, dim=-1))
        out = torch.matmul(attn, V).transpose(1, 2).contiguous().view(B, -1, self.n_heads * self.d_k)
        return self.w_o(out)

class PositionwiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_model, d_ff), nn.ReLU(), nn.Dropout(dropout), nn.Linear(d_ff, d_model))
    def forward(self, x): return self.net(x)

class EncoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.ff = PositionwiseFeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model); self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x, mask):
        x = self.norm1(x + self.dropout(self.self_attn(x, x, x, mask)))
        return self.norm2(x + self.dropout(self.ff(x)))

class Encoder(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, d_ff, n_layers, dropout=0.1, max_len=100):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.pos_enc = PositionalEncoding(d_model, max_len)
        self.layers = nn.ModuleList([EncoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)])
        self.dropout = nn.Dropout(dropout); self.d_model = d_model
    def forward(self, src, mask):
        x = self.embed(src) * math.sqrt(self.d_model)
        x = self.dropout(self.pos_enc(x))
        for layer in self.layers: x = layer(x, mask)
        return x

class DecoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.cross_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.ff = PositionwiseFeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model); self.norm2 = nn.LayerNorm(d_model); self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x, enc_out, src_mask, tgt_mask):
        x = self.norm1(x + self.dropout(self.self_attn(x, x, x, tgt_mask)))
        x = self.norm2(x + self.dropout(self.cross_attn(x, enc_out, enc_out, src_mask)))
        return self.norm3(x + self.dropout(self.ff(x)))

class Decoder(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, d_ff, n_layers, dropout=0.1, max_len=100):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.pos_enc = PositionalEncoding(d_model, max_len)
        self.layers = nn.ModuleList([DecoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)])
        self.fc_out = nn.Linear(d_model, vocab_size)
        self.dropout = nn.Dropout(dropout); self.d_model = d_model
    def forward(self, tgt, enc_out, src_mask, tgt_mask):
        x = self.embed(tgt) * math.sqrt(self.d_model)
        x = self.dropout(self.pos_enc(x))
        for layer in self.layers: x = layer(x, enc_out, src_mask, tgt_mask)
        return self.fc_out(x)

class Transformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model=256, n_heads=8, d_ff=512, n_layers=3, dropout=0.1, max_len=100, pad_idx=0):
        super().__init__()
        self.encoder = Encoder(src_vocab_size, d_model, n_heads, d_ff, n_layers, dropout, max_len)
        self.decoder = Decoder(tgt_vocab_size, d_model, n_heads, d_ff, n_layers, dropout, max_len)
        self.pad_idx = pad_idx
    def make_src_mask(self, src): return (src != self.pad_idx).unsqueeze(1).unsqueeze(2)
    def make_tgt_mask(self, tgt):
        pad_mask = (tgt != self.pad_idx).unsqueeze(1).unsqueeze(2)
        L = tgt.size(1)
        sub_mask = torch.tril(torch.ones((L, L), device=tgt.device)).bool()
        return pad_mask & sub_mask
    def forward(self, src, tgt):
        src_mask = self.make_src_mask(src); tgt_mask = self.make_tgt_mask(tgt)
        enc_out = self.encoder(src, src_mask)
        return self.decoder(tgt, enc_out, src_mask, tgt_mask)

model = Transformer(src_vocab_size=len(stoi_en), tgt_vocab_size=len(stoi_es), **model_cfg, pad_idx=PAD_IDX).to(DEVICE)
model.load_state_dict(torch.load(os.path.join(MODEL_DIR_1M, "best_model.pth"), map_location=DEVICE))
model.eval()
print(f"Modelo cargado (val loss 3.5827, época 14). Vocab EN: {len(stoi_en):,} | ES: {len(stoi_es):,}")

Mounted at /content/drive
Dispositivo: cpu
Modelo cargado (val loss 3.5827, época 14). Vocab EN: 28,160 | ES: 37,161


In [4]:
# =========================
# TRADUCCIÓN (beam search) + PRUEBA DE VELOCIDAD
# =========================
import re
TOKEN_RE = re.compile(r"\w+(?:'\w+)*|[^\w\s]")
def tokenize(text):
    return TOKEN_RE.findall(str(text))

MAX_LEN_GEN = model_cfg.get("max_len", 32)  # tope de generación

def translate_beam_search(sentence, beam_size=3, max_len=32):
    model.eval()
    tokens = tokenize(sentence)[:30]
    src_ids = [SOS_IDX] + [stoi_en.get(t, UNK_IDX) for t in tokens] + [EOS_IDX]
    src_tensor = torch.tensor([src_ids], dtype=torch.long).to(DEVICE)
    src_mask = model.make_src_mask(src_tensor)

    with torch.no_grad():
        enc_out = model.encoder(src_tensor, src_mask)

    beams = [([SOS_IDX], 0.0)]
    for _ in range(max_len):
        candidates = []
        all_done = True
        for ids, log_prob in beams:
            if ids[-1] == EOS_IDX:
                candidates.append((ids, log_prob)); continue
            all_done = False
            tgt_tensor = torch.tensor([ids], dtype=torch.long).to(DEVICE)
            tgt_mask = model.make_tgt_mask(tgt_tensor)
            with torch.no_grad():
                out = model.decoder(tgt_tensor, enc_out, src_mask, tgt_mask)
            log_probs = F.log_softmax(out[0, -1], dim=-1)
            topk_probs, topk_ids = log_probs.topk(beam_size)
            for k in range(beam_size):
                candidates.append((ids + [topk_ids[k].item()], log_prob + topk_probs[k].item()))
        if all_done:
            break
        candidates.sort(key=lambda x: x[1], reverse=True)
        beams = candidates[:beam_size]

    best_ids, _ = beams[0]
    words = [itos_es.get(i, "<UNK>") for i in best_ids[1:] if i != EOS_IDX]
    return " ".join(words)

# --- Prueba rápida con 5 oraciones manuales ---
print("=== Prueba manual ===")
for s in ["I love you.", "What time is it?", "Where is the bathroom?", "Thank you very much.", "I don't understand."]:
    print(f"EN: {s}\nES: {translate_beam_search(s)}\n")

# --- Medición de velocidad sobre el test set real ---
test_pairs = []
with open(os.path.join(SPLIT_DIR_1M, "test.tsv"), encoding="utf-8") as f:
    for line in f:
        parts = line.rstrip("\n").split("\t")
        if len(parts) >= 2:
            test_pairs.append((parts[0], parts[1]))
print(f"Test set total: {len(test_pairs):,} pares")

t0 = time.time()
N_PROBE = 20
for src, _ in test_pairs[:N_PROBE]:
    translate_beam_search(src)
dt = time.time() - t0
print(f"\n{N_PROBE} traducciones: {dt:.1f}s -> {dt/N_PROBE:.2f}s/oración")
print(f"Estimado para 100 ejemplos: {dt/N_PROBE*100/60:.1f} min")
print(f"Estimado para el test completo ({len(test_pairs):,}): {dt/N_PROBE*len(test_pairs)/60:.1f} min ({dt/N_PROBE*len(test_pairs)/3600:.1f} h)")

=== Prueba manual ===
EN: I love you.
ES: Te quiero .

EN: What time is it?
ES: ¿ Qué hora es ?

EN: Where is the bathroom?
ES: ¿ Dónde está el baño ?

EN: Thank you very much.
ES: Muchas gracias .

EN: I don't understand.
ES: No entiendo .

Test set total: 44,004 pares

20 traducciones: 5.7s -> 0.28s/oración
Estimado para 100 ejemplos: 0.5 min
Estimado para el test completo (44,004): 208.5 min (3.5 h)


In [6]:
# =========================
# EVALUACIÓN: BLEU + 100+ EJEMPLOS (muestra de 500, CPU)
# =========================
import random
SEED = 42
random.seed(SEED)

N_EVAL = 500
sample_pairs = random.sample(test_pairs, N_EVAL)

results = []
t0 = time.time()
for i, (src, ref) in enumerate(sample_pairs):
    pred = translate_beam_search(src)
    bleu = sentence_bleu_smoothed(ref, pred)
    results.append({"english": src, "spanish_reference": ref, "spanish_predicted": pred, "bleu": bleu})
    if (i + 1) % 100 == 0:
        print(f"  ...{i+1}/{N_EVAL} ({(time.time()-t0)/60:.1f} min)")

df_eval = pd.DataFrame(results)
dt = time.time() - t0
print(f"\nTiempo total: {dt/60:.1f} min")

# --- BLEU a nivel de corpus (métrica principal) ---
corpus_score, precisions, bp = corpus_bleu(df_eval["spanish_reference"].tolist(), df_eval["spanish_predicted"].tolist())
print(f"\n=== RESULTADOS (muestra de {N_EVAL}) ===")
print(f"BLEU de corpus          : {corpus_score:.4f}")
print(f"Precisiones por n-grama : {[round(p,3) for p in precisions]}")
print(f"Brevity penalty         : {bp:.4f}")
print(f"BLEU promedio por oración (referencia): {df_eval['bleu'].mean():.4f}")

# --- Clasificación cualitativa ---
def clasificar(b):
    if b >= 0.7: return "bueno"
    if b >= 0.3: return "aceptable"
    return "error"
df_eval["categoria"] = df_eval["bleu"].map(clasificar)
print("\nDistribución:", df_eval["categoria"].value_counts().to_dict())

print("\n=== 10 mejores ===")
display(df_eval.nlargest(10, "bleu")[["english", "spanish_reference", "spanish_predicted", "bleu"]])
print("\n=== 10 peores ===")
display(df_eval.nsmallest(10, "bleu")[["english", "spanish_reference", "spanish_predicted", "bleu"]])

# --- Guardar ---
REPORTE_DIR = os.path.join(PROJ, "reporte")
os.makedirs(REPORTE_DIR, exist_ok=True)
df_eval.to_csv(os.path.join(REPORTE_DIR, "resultados_test_opensubtitles_muestra500.csv"), index=False)
print(f"\nGuardado en: {REPORTE_DIR}")

  ...100/500 (0.6 min)
  ...200/500 (1.1 min)
  ...300/500 (1.7 min)
  ...400/500 (2.4 min)
  ...500/500 (2.9 min)

Tiempo total: 2.9 min

=== RESULTADOS (muestra de 500) ===
BLEU de corpus          : 0.1695
Precisiones por n-grama : [0.51, 0.235, 0.127, 0.069]
Brevity penalty         : 0.9406
BLEU promedio por oración (referencia): 0.2881

Distribución: {'error': 331, 'aceptable': 141, 'bueno': 28}

=== 10 mejores ===


,english,spanish_reference,spanish_predicted,bleu
25,What makes you think so?,¿Qué te hace pensar eso?,¿ Qué te hace pensar eso ?,1.0
41,And there are no songs.,Y no hay canciones.,Y no hay canciones .,1.0
72,"""I love you.""","""Te quiero.""",""" Te quiero . """,1.0
111,I do not need to.,No necesito hacerlo.,No necesito hacerlo .,1.0
114,Don't feel too bad.,No te sientas muy mal.,No te sientas muy mal .,1.0
157,Are you feeling all right?,¿Te sientes bien?,¿ Te sientes bien ?,1.0
255,"No, this way. Oh!","No, por aquí.","No , por aquí .",1.0
257,"Okay, okay.",Está bien.,Está bien .,1.0
291,l want him on my show.,Lo quiero en mi programa.,Lo quiero en mi programa .,1.0
299,And you found a coin?,¿Y encontraste una moneda?,¿ Y encontraste una moneda ?,1.0



=== 10 peores ===


,english,spanish_reference,spanish_predicted,bleu
135,Listen.,"Ella está bien, ¿de acuerdo?",Escucha .,0.020325
184,"My coin, sir! - Go away.","Jean Valjean, no creo en el poder del dinero.",¡ Vete !,0.024089
160,Where do you expect we're gonna find bird bait...,¿Cebo para pájaros? ¿De dónde pensáis sacar ce...,¿ Dónde vas a encontrar el pájaro ?,0.025328
161,Just push and make way for yourself.,Impulsa y da el paso por ti mismo. Esta es la ...,Solo haz que te <UNK> .,0.045088
453,"I think maybe that would be in order, but you ...","Creo que tal vez eso sería en orden, pero no t...",No creo que sea necesario .,0.051225
228,That song and dance number attacking Hilary Cl...,Eso de atacar a Hillary Clinton fue muy contro...,¿ Esa canción de <UNK> y <UNK> el número de ba...,0.051997
159,Please check the number and try your call again.,Por favor revise el número y marque de nuevame...,"Por favor , llama al número de nuevo .",0.055279
479,You are hereby quarantined until such time as ...,Por la presente están en cuarentena hasta que ...,Estás tan segura como para disparar y disparar...,0.059153
180,"I've thrown in the Last reserve, Khokhlov's ar...",En los flancos de la división se ha creado una...,"La última vez que he <UNK> en la <UNK> , <UNK> .",0.064658
132,I just can't figure out how to get my whites w...,Es sólo que no se me ocurre cómo hacer mi ropa...,No puedo averiguar cómo sacar mis blancos .,0.068848



Guardado en: /content/drive/MyDrive/Proyecto_traductor/reporte


In [7]:
# =========================
# BLEU POR TRAMO DE LONGITUD (confirma el patrón observado)
# =========================
df_eval["ref_len"] = df_eval["spanish_reference"].map(lambda s: len(tokenize(s)))
bins = [0, 5, 10, 15, 20, 30]
labels = ["1-5", "6-10", "11-15", "16-20", "21-30"]
df_eval["tramo"] = pd.cut(df_eval["ref_len"], bins=bins, labels=labels)

print("=== BLEU promedio por tramo de longitud de referencia ===")
display(df_eval.groupby("tramo", observed=True).agg(
    n=("bleu", "size"), bleu_medio=("bleu", "mean"),
    pct_error=("categoria", lambda s: (s == "error").mean() * 100)
).round(3))

=== BLEU promedio por tramo de longitud de referencia ===


,n,bleu_medio,pct_error
tramo,,,
1-5,164,0.379,43.902
6-10,218,0.284,70.183
11-15,74,0.193,89.189
16-20,32,0.141,87.500
21-30,12,0.108,100.000
